# Iris 分類器の実装と評価

このノートブックでは、TDD で実装した Iris 分類器を使用して、モデルの訓練と評価を行います。

## 1. セットアップ

必要なライブラリをロードします。

In [ ]:
(require '[ml-tdd-project.ml.iris-classifier :as iris])
(require '[tablecloth.api :as tc])

## 2. データの読み込み

Iris データセットを読み込みます。

In [ ]:
(def dataset (iris/load-data "../resources/data/iris.csv"))

;; データの基本情報を表示
(println "データセットの形状:")
(println (str "  行数: " (tc/row-count dataset)))
(println (str "  列数: " (tc/column-count dataset)))
(println (str "\n列名: " (tc/column-names dataset)))

## 3. データの確認

最初の数行を表示してデータを確認します。

In [ ]:
;; 最初の10行を表示
(tc/head dataset 10)

## 4. クラスの分布

各種 (species) のサンプル数を確認します。

In [ ]:
(-> dataset
    (tc/group-by :species)
    (tc/aggregate {:count tc/row-count}))

## 5. データの分割

データを訓練用 (80%) とテスト用 (20%) に分割します。

In [ ]:
;; シャッフルしてからランダムに分割
(def shuffled-indices (shuffle (vec (range (tc/row-count dataset)))))
(def train-size (int (* 0.8 (tc/row-count dataset))))
(def train-indices (vec (take train-size shuffled-indices)))
(def test-indices (vec (drop train-size shuffled-indices)))
(def train-data (tc/select-rows dataset train-indices))
(def test-data (tc/select-rows dataset test-indices))

(println (str "訓練データ: " (tc/row-count train-data) " 行"))
(println (str "テストデータ: " (tc/row-count test-data) " 行"))

## 6. モデルの作成と訓練

決定木分類器を作成し、訓練データで訓練します。

In [ ]:
;; 分類器の作成（最大ノード数 20）
(def classifier (iris/create-classifier {:max-nodes 20}))
(println (str "最大ノード数: " (:max-nodes classifier)))

;; モデルの訓練
(def trained-classifier (iris/train classifier train-data))
(println "\n訓練完了")

## 7. 予測の実行

テストデータで予測を実行します。

In [ ]:
(def predictions (iris/predict trained-classifier test-data))
(def actual-labels (vec (tc/column test-data :species)))

(println "予測結果（最初の10件）:")
(doseq [i (range (min 10 (count predictions)))]
  (let [pred (nth predictions i)
        actual (nth actual-labels i)
        match (if (= pred actual) "✓" "✗")]
    (println (str (inc i) ". 予測: " pred ", 実際: " actual " " match))))

## 8. モデルの評価

正解率を計算します。

In [ ]:
(def accuracy (iris/evaluate trained-classifier test-data))
(println (str "正解率: " (format "%.2f" (* 100 accuracy)) "%"))

## 9. 混同行列

予測と実際のラベルの対応を混同行列で確認します。

In [ ]:
(def species-names (sort (distinct actual-labels)))
(def confusion-matrix
  (reduce (fn [m [pred actual]]
            (update m [actual pred] (fnil inc 0)))
          {}
          (map vector predictions actual-labels)))

(println "\n混同行列:")
(print "           ")
(doseq [s species-names]
  (print (format "%-12s" s)))
(println)
(doseq [actual species-names]
  (print (format "%-10s " actual))
  (doseq [pred species-names]
    (print (format "%-12d" (get confusion-matrix [actual pred] 0))))
  (println))

## 10. 異なるパラメータでの実験

異なる最大ノード数でモデルを訓練し、性能を比較します。

In [ ]:
(def max-nodes-list [2 5 10 20 50])

(println "最大ノード数と正解率の関係:")
(doseq [max-nodes max-nodes-list]
  (let [clf (iris/create-classifier {:max-nodes max-nodes})
        trained (iris/train clf train-data)
        acc (iris/evaluate trained test-data)]
    (println (str "  max-nodes=" max-nodes ": " (format "%.2f" (* 100 acc)) "%"))))

## まとめ

このノートブックでは以下を実施しました：

1. Iris データセットの読み込みと確認
2. データの訓練用とテスト用への分割
3. 決定木分類器の作成と訓練
4. テストデータでの予測と評価
5. 混同行列による詳細な分析
6. パラメータチューニングの実験

TDD で実装した分類器が正常に動作することを確認できました。